# Module 10 – Lab 1: Implement Observability and Tracing for a RAG Application

**Audience:** non-technical learners. No programming experience is needed.
**Time:** about 60 minutes. **Before you start:** complete the setup steps in `README.md` (Docker Desktop and your `.env` file).

### What you will do
1. Start **Jaeger** (a trace viewer) with one Docker command.
2. Run a small **RAG** (Retrieval-Augmented Generation) assistant that answers IT questions.
3. Use **OpenTelemetry** to record every step the assistant takes.
4. Open the **Jaeger UI** and read the trace for each question.
5. Trigger a controlled error and find the step that caused it.

---
## Section 1 – Introduction to Observability

**What it is:** *observability* means understanding what is happening inside an application by looking at the data it produces.

**Real-world example:** a parcel-tracking page shows every checkpoint (sorting hub, out for delivery, delivered) and the time spent at each. If a parcel is late, you can see exactly where it waited. Tracing does the same job for software.

**Why it matters for Generative AI:**
- An AI answer takes time and money. You need to know **which step was slow**.
- A poor answer can come from the document search, the prompt, or the model. Tracing shows which one.
- An error can happen in one step while the rest of the application looks fine.

| Term | Plain-English meaning |
|---|---|
| **Distributed tracing** | Following one request step by step through every part of the application that handled it. |
| **Trace** | The complete record of one request (here: one question and its answer). |
| **Span** | One step inside a trace, with a name, a start time, an end time and details. |
| **Parent / child span** | A parent is a bigger step. Its children are the smaller steps inside it. |
| **OpenTelemetry (OTel)** | A free, open-standard toolkit that creates spans and sends them to a tracing tool. |
| **Jaeger** | A free, open-source tool that stores traces and shows them as timelines in a web browser. |

**The trace you will build for each question:**

```text
RAG Application                    (the whole run)
  └── User Query                   (query processing)
        ├── Document Retrieval     (search the knowledge base)
        ├── Prompt Construction    (build the prompt)
        ├── LLM Invocation         (call the AI model)
        └── Response Generation    (finish the answer)
```

> **Training note:** this is a simplified RAG implementation for learning. It is **not** a production retrieval architecture.

## Section 2 – Environment Setup

**What this section does:** imports the Python libraries, loads your OpenAI API key from the local `.env` file, and sets up OpenTelemetry so it can send traces to Jaeger.

**Why it is required:** every later cell depends on these settings.

**Before you run this cell:** make sure your `.env` file exists in this project folder (README, step 3). The key stays in that file. This notebook never prints it.

The libraries below are already listed in `requirements.txt`, so you do **not** install them one by one:

| Library | Purpose in this lab |
|---|---|
| `python-dotenv` | Reads your API key from the `.env` file. |
| `langchain-core` | LangChain building blocks: documents, prompts and runnables. |
| `langchain-openai` | Connects LangChain to the OpenAI chat model. |
| `opentelemetry-api` | The interface used to create spans. |
| `opentelemetry-sdk` | Collects spans, creates trace and span IDs, and measures time. |
| `opentelemetry-exporter-otlp-proto-grpc` | Sends finished spans to Jaeger using the OTLP protocol. |

In [ ]:
# ---- Standard library ----
import os                # reads environment variables (such as the API key)
import re                # finds words in text
import socket            # checks whether a network port is open
import time              # measures how long steps take
import urllib.request    # checks whether the Jaeger web page answers
import uuid              # creates short random IDs

# ---- Secrets and LangChain ----
from dotenv import load_dotenv                        # reads the .env file
from langchain_core.documents import Document          # LangChain's standard document object
from langchain_core.prompts import ChatPromptTemplate  # prompt with placeholders
from langchain_core.runnables import RunnableLambda    # wraps a Python function so it has .invoke()
from langchain_openai import ChatOpenAI                # connector for the OpenAI chat model

# ---- OpenTelemetry (tracing) ----
from opentelemetry import trace                                                     # creates tracers and spans
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter  # sends spans to Jaeger
from opentelemetry.sdk.resources import Resource                                    # labels attached to every span
from opentelemetry.sdk.trace import TracerProvider                                  # collects spans
from opentelemetry.sdk.trace.export import BatchSpanProcessor                       # sends spans in batches

# ---- Load the API key safely ----
load_dotenv()  # copies the values in .env (if the file exists) into environment variables
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key or api_key == "your_openai_api_key_here":
    raise RuntimeError(
        "OPENAI_API_KEY was not found. Copy .env.example to a new file named .env, "
        "paste your key after OPENAI_API_KEY=, save the file, and run this cell again."
    )
print("OpenAI API key found (the value is hidden for safety).")

# ---- Configure OpenTelemetry to send traces to Jaeger ----
SERVICE_NAME = "module10-rag-app"               # the name shown in Jaeger's Service dropdown
JAEGER_OTLP_ENDPOINT = "http://localhost:4317"  # where Jaeger receives traces (OTLP gRPC)
JAEGER_UI_URL = "http://localhost:16686"        # where you view traces in the browser

tracer_provider = TracerProvider(resource=Resource.create({"service.name": SERVICE_NAME}))
span_exporter = OTLPSpanExporter(endpoint=JAEGER_OTLP_ENDPOINT, insecure=True)  # local Jaeger uses no TLS
tracer_provider.add_span_processor(BatchSpanProcessor(span_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer("module10.rag")       # the tracer creates every span in this lab

print("OpenTelemetry is ready. Traces will be sent to", JAEGER_OTLP_ENDPOINT)

### What the setup cell did
- **`load_dotenv()`** reads `OPENAI_API_KEY` from `.env`. If the key is missing, the cell stops with a clear message instead of failing later.
- **`SERVICE_NAME`** is the label Jaeger shows in its *Service* dropdown. Every span from this notebook is tagged with it.
- **`JAEGER_OTLP_ENDPOINT = "http://localhost:4317"`** is the address where Jaeger receives traces. Port **4317** is the OTLP gRPC port. The Python app sends data to it. You never open this address in a browser.
- **`TracerProvider`** is the OpenTelemetry object that creates spans. **`BatchSpanProcessor`** collects finished spans and sends them in small batches, the way production systems usually do.
- Run this cell **once** per kernel session. If you run it again, you may see a harmless warning that the tracer provider cannot be overridden. The first setup keeps working. To start fresh, use *Restart Kernel*.

## Section 3 – Start and Verify Jaeger

**What this section does:** starts Jaeger on your computer and checks that this notebook can reach it.

**Why it is required:** Jaeger receives the traces from the notebook and shows them in a browser. Without Jaeger the notebook still runs, but you cannot see any traces.

**Start Jaeger.** Run these steps in a terminal opened in the project folder (not in this notebook):
1. Open **Docker Desktop** and wait until it shows that the engine is running.
2. Run: `docker compose up -d`
3. Open **http://localhost:16686** in your browser. You should see the Jaeger page.

**How the Python app communicates with Jaeger:**

```text
Notebook (Python) ──traces via OTLP, port 4317──▶  Jaeger  ◀──browser, port 16686──  You
```

In [ ]:
# Check 1: can we reach the Jaeger web page?
try:
    with urllib.request.urlopen(JAEGER_UI_URL, timeout=3) as response:
        print("Jaeger UI is running at", JAEGER_UI_URL, "(HTTP status", response.status, ")")
except OSError as error:
    print("Jaeger UI is NOT reachable at", JAEGER_UI_URL)
    print("Fix: open Docker Desktop, then run 'docker compose up -d' in the project folder.")
    print("Details:", error)

# Check 2: is the OTLP port open? This is the port the notebook sends traces to.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as connection:
    connection.settimeout(2)
    port_result = connection.connect_ex(("localhost", 4317))

if port_result == 0:
    print("OTLP port 4317 is open. Traces can be sent to Jaeger.")
else:
    print("OTLP port 4317 is closed. Start Jaeger before running Section 8.")

### What the check showed
- **"Jaeger UI is running"** means the web page answers, so you can open it in your browser.
- **"OTLP port 4317 is open"** means the notebook can send traces to Jaeger.
- If either line says *not* reachable, open Docker Desktop, run `docker compose up -d` again, and re-run this cell.
- Jaeger looks **empty** until you run the RAG application in Section 8. This is expected.
- Jaeger keeps traces in memory only. When you run `docker compose down`, the traces are removed.

## Section 4 – Create a Sample Knowledge Base

**What this section does:** creates seven short IT Operations documents inside the notebook. **All documents are fictional** and contain no real company or employee data.

**Why it is required:** the assistant needs documents to search. Keeping them in the notebook means there are no extra files to load.

**How it is used:** each question is compared with every document. The best matches are sent to the AI model as context, so the answer is based on these documents.

**Training note:** the search uses simple word matching so that learners can follow every step. It is **not** a production retrieval architecture and does not use a vector database.

In [ ]:
# Each Document holds the text (page_content) and labels (metadata) such as an ID and a title.
KNOWLEDGE_BASE = [
    Document(
        page_content=(
            "Password Reset Procedure. If a user is locked out of their account or has forgotten "
            "their password, they can use the self-service reset page. If that fails, the service desk "
            "resets the password. A temporary password must be changed at the first login."
        ),
        metadata={"doc_id": "KB-001", "title": "Password Reset Procedure"},
    ),
    Document(
        page_content=(
            "VPN Connectivity Troubleshooting. If the VPN keeps disconnecting, check your internet connection, "
            "restart the VPN client, confirm your VPN access is active, and install the latest client update. "
            "If the problem continues, raise a ticket with the service desk."
        ),
        metadata={"doc_id": "KB-002", "title": "VPN Connectivity Troubleshooting"},
    ),
    Document(
        page_content=(
            "Laptop Performance Issues. If a laptop is running slowly, close unused applications, restart "
            "the device, check that the hard drive has at least 15 percent free space, and run the approved "
            "maintenance tool. Laptops older than five years may need a hardware review."
        ),
        metadata={"doc_id": "KB-003", "title": "Laptop Performance Issues"},
    ),
    Document(
        page_content=(
            "Email Access Problems. If email will not send or receive, check your network connection, confirm "
            "your password is current, restart the mail application, and check whether the mailbox is full. "
            "Requests for access to a shared mailbox go to the service desk."
        ),
        metadata={"doc_id": "KB-004", "title": "Email Access Problems"},
    ),
    Document(
        page_content=(
            "Network Connectivity Troubleshooting. If there is no internet access, check the network cable or "
            "Wi-Fi connection, renew the IP address, and check whether colleagues nearby are also affected. "
            "Report a wide outage as a server or network incident."
        ),
        metadata={"doc_id": "KB-005", "title": "Network Connectivity Troubleshooting"},
    ),
    Document(
        page_content=(
            "Server Incident Escalation. Severity P1 means a service outage affecting many users. "
            "Escalate P1 incidents to the on-call infrastructure team within 15 minutes. Record the incident "
            "ticket number in every update."
        ),
        metadata={"doc_id": "KB-006", "title": "Server Incident Escalation"},
    ),
    Document(
        page_content=(
            "IT Service Desk Procedures. The service desk accepts requests by ticket or phone during business "
            "hours, Monday to Friday, 8 AM to 6 PM. Never share a password by email or chat. "
            "All contact details in this training are fictional."
        ),
        metadata={"doc_id": "KB-007", "title": "IT Service Desk Procedures"},
    ),
]

print("Knowledge base created with", len(KNOWLEDGE_BASE), "fictional documents:")
for doc in KNOWLEDGE_BASE:
    print("  ", doc.metadata["doc_id"], "-", doc.metadata["title"])

## Section 5 – Build a Simple RAG Application

**What this section does:** builds the four parts of a RAG assistant, **without tracing yet**:
1. **Retrieve**: find the documents most relevant to the question.
2. **Build the prompt**: combine those documents and the question into one instruction.
3. **Call the LLM**: send the prompt to the OpenAI model.
4. **Return the answer.**

**Why it is required:** you must understand what the application normally does before you can trace it. In Section 7 we wrap these same functions with tracing.

**Note:** `RunnableLambda` turns a normal Python function into a LangChain component that you run with `.invoke()`. That is the current LangChain style.

In [ ]:
TOP_K = 3                   # how many knowledge-base documents to retrieve for each question
MODEL_NAME = "gpt-6-luna"   # OpenAI's current small chat model. Change this if your account does not offer it.

# Common words that do not help the search (for example "the", "how", "my")
STOP_WORDS = {
    "a", "am", "an", "and", "at", "be", "can", "do", "for", "how", "i", "if", "in", "is",
    "it", "me", "my", "of", "on", "or", "out", "should", "the", "this", "to", "very", "what",
}

def retrieve_documents(question):
    """Return the TOP_K documents that share the most words with the question."""
    question_words = set(re.findall(r"[a-z]+", question.lower())) - STOP_WORDS
    scored = []
    for doc in KNOWLEDGE_BASE:
        doc_words = set(re.findall(r"[a-z]+", doc.page_content.lower()))
        score = len(question_words & doc_words)   # number of shared words
        scored.append((score, doc))
    scored.sort(key=lambda pair: pair[0], reverse=True)   # best match first
    return [doc for score, doc in scored[:TOP_K] if score > 0]

retriever = RunnableLambda(retrieve_documents)   # LangChain component: use retriever.invoke(question)

def format_context(documents):
    """Join the retrieved documents into one block of text for the prompt."""
    lines = []
    for doc in documents:
        lines.append(f"[{doc.metadata['doc_id']}] {doc.page_content}")
    return "\n".join(lines)

PROMPT_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", "You are an IT Operations assistant. Answer the question using ONLY the context provided. "
               "If the context does not contain the answer, say you do not know."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def build_prompt(documents, question):
    """Fill the prompt template with the retrieved context and the user's question."""
    return PROMPT_TEMPLATE.invoke({"context": format_context(documents), "question": question})

llm = ChatOpenAI(model=MODEL_NAME)   # connector that sends prompts to the OpenAI model

def generate_answer(prompt_value):
    """Send the prompt to the LLM and return the model's reply."""
    return llm.invoke(prompt_value)

def plain_rag(question):
    """Simple RAG without tracing: retrieve, build the prompt, call the LLM, return the text."""
    documents = retriever.invoke(question)
    prompt_value = build_prompt(documents, question)
    reply = generate_answer(prompt_value)
    return reply.content

# Quick test: which documents were retrieved, and what did the assistant answer?
test_question = "How do I reset my password if I am locked out?"
print("Question:", test_question)
print("Retrieved document IDs:", [doc.metadata["doc_id"] for doc in retriever.invoke(test_question)])
print("Answer:", plain_rag(test_question))

### What the test showed
- The **retrieved document IDs** (for example `KB-001`) show which documents the search selected. The top match should be the password procedure.
- The **answer** comes from those documents only. The system prompt tells the model to say "I do not know" when the documents do not contain the answer.
- Wording can still vary slightly from run to run. That is normal for language models; the source documents stay the same.

## Section 6 – Introduce OpenTelemetry Tracing

**What this section does:** introduces the building blocks of tracing with a small toy example, before we touch the RAG application.

**Key ideas:**
- **Trace:** one complete story, such as one question from start to finish. Every trace has a **Trace ID** (a 32-character code).
- **Span:** one step in that story. Every span has a **Span ID** (a 16-character code).
- **Parent and child spans:** when a span starts while another span is still running, it becomes that span's **child**. The child shares the parent's Trace ID and records the parent's Span ID.
- **Attributes:** extra facts attached to a span, such as a document count or a model name.
- **Duration:** the time between a span's start and end. Jaeger draws it as a bar.

The demo below creates one parent span with two child spans, then prints their IDs and durations.

In [ ]:
# One parent span with two child spans inside it.
with tracer.start_as_current_span("Demo: Parent Operation") as parent_span:
    parent_context = parent_span.get_span_context()
    print("Parent span -> Trace ID:", format(parent_context.trace_id, "032x"))
    print("Parent span -> Span ID: ", format(parent_context.span_id, "016x"))

    # Child span A: pretend to work for 0.2 seconds
    with tracer.start_as_current_span("Demo: Child Step A") as child_a:
        child_a.set_attribute("demo.step", "A")
        time.sleep(0.2)

    # Child span B: pretend to work for 0.5 seconds
    with tracer.start_as_current_span("Demo: Child Step B") as child_b:
        child_b.set_attribute("demo.step", "B")
        time.sleep(0.5)

# Once a span has ended, its start and end times give its duration (in nanoseconds).
for name, span in [("Parent", parent_span), ("Child A", child_a), ("Child B", child_b)]:
    duration_ms = (span.end_time - span.start_time) / 1_000_000
    print(f"{name:<8} duration: {duration_ms:.0f} ms")

print("Child A's parent span ID:", format(child_a.parent.span_id, "016x"))

### How to read the demo output
- Both child spans show the **same Trace ID** as the parent, because they belong to the same story.
- **Child A's parent span ID** equals the parent's **Span ID**. This is the parent-child link. Jaeger shows it as indentation.
- The durations come from the sleep times: about 200 ms and 500 ms. The parent lasts about as long as its children together.
- The demo spans were also sent to Jaeger under the service `module10-rag-app`.

## Section 7 – Instrument the RAG Application

**What this section does:** wraps each step of the RAG application in a span. Every question then produces one trace with a clear structure.

**The trace this creates:**

| Span | Recorded details (safe metadata only) |
|---|---|
| RAG Application | Query ID, model name, final status |
| User Query | Query ID, query length in characters |
| Document Retrieval | Number of documents found, document IDs, top-K setting |
| Prompt Construction | Number of context documents, prompt length |
| LLM Invocation | Provider, model name, input and output token counts (only when OpenAI returns them) |
| Response Generation | Finish reason, response length, status |

**Security rules used in this lab:**
- **No API key** is written to any span.
- **No full question, prompt or answer** is written to any span. A short query ID and counts are recorded instead.
- **Only fictional data** is used.

**Why these rules matter:** trace data is usually stored, shared and read by many people. If a trace contained personal data or a secret, everyone with access to Jaeger would see it. Production systems follow the same rule.

In [ ]:
def answer_question(question, simulate_failure=False):
    """Answer one question. Each step runs inside its own span, so Jaeger shows the whole story."""
    query_id = uuid.uuid4().hex[:8]   # short random ID. We record this instead of the question text.
    started = time.perf_counter()     # start the stopwatch for the execution time

    # Parent span: the whole RAG run. Everything inside this "with" block becomes its child.
    with tracer.start_as_current_span("RAG Application") as app_span:
        app_span.set_attribute("rag.query_id", query_id)
        app_span.set_attribute("gen_ai.request.model", MODEL_NAME)
        trace_id = format(app_span.get_span_context().trace_id, "032x")   # 32-character Trace ID
        print("Trace ID for this question:", trace_id)

        # Child span: query processing. The four steps below are its children.
        with tracer.start_as_current_span("User Query") as query_span:
            query_span.set_attribute("rag.query_id", query_id)
            query_span.set_attribute("rag.query_length_chars", len(question))

            # Step 1: Document Retrieval
            with tracer.start_as_current_span("Document Retrieval") as span:
                if simulate_failure:   # used only in Section 10 to show how an error looks in Jaeger
                    raise RuntimeError("Simulated failure: the knowledge base index is unavailable")
                documents = retriever.invoke(question)
                doc_ids = [doc.metadata["doc_id"] for doc in documents]
                span.set_attribute("retrieval.top_k", TOP_K)
                span.set_attribute("retrieval.documents_found", len(documents))
                span.set_attribute("retrieval.document_ids", ", ".join(doc_ids))

            # Step 2: Prompt Construction
            with tracer.start_as_current_span("Prompt Construction") as span:
                prompt_value = build_prompt(documents, question)
                span.set_attribute("prompt.context_documents", len(documents))
                span.set_attribute("prompt.length_chars", len(prompt_value.to_string()))

            # Step 3: LLM Invocation (usually the slowest step)
            with tracer.start_as_current_span("LLM Invocation") as span:
                span.set_attribute("gen_ai.provider.name", "openai")
                span.set_attribute("gen_ai.request.model", MODEL_NAME)
                llm_reply = generate_answer(prompt_value)
                usage = llm_reply.usage_metadata   # token counts from OpenAI, or None if not returned
                if usage:
                    span.set_attribute("gen_ai.usage.input_tokens", usage["input_tokens"])
                    span.set_attribute("gen_ai.usage.output_tokens", usage["output_tokens"])

            # Step 4: Response Generation (finish the answer and record its status)
            with tracer.start_as_current_span("Response Generation") as span:
                answer_text = llm_reply.content.strip()
                finish_reason = llm_reply.response_metadata.get("finish_reason", "unknown")
                span.set_attribute("gen_ai.response.finish_reasons", [finish_reason])
                span.set_attribute("response.length_chars", len(answer_text))
                span.set_attribute("response.status", "success")

        app_span.set_attribute("rag.status", "success")

    tracer_provider.force_flush(timeout_millis=5000)   # send this trace to Jaeger now
    elapsed_seconds = time.perf_counter() - started
    return {
        "query_id": query_id,
        "trace_id": trace_id,
        "documents_found": len(documents),
        "answer": answer_text,
        "seconds": elapsed_seconds,
    }

### How the instrumented function works
- **`tracer.start_as_current_span("...")`** starts a span. Everything inside its indented `with` block becomes its **child**. The indentation in the code *is* the parent-child structure.
- **`set_attribute(...)`** adds one safe detail to the current span, such as the number of documents found.
- **`time.perf_counter()`** is a stopwatch for the total execution time that the notebook displays. Jaeger shows the duration of every span separately.
- **Token counts** come from `usage_metadata`. If OpenAI does not return them, nothing is recorded. We never guess.
- **`raise` inside the retrieval span** is used only for the Section 10 demonstration. Normal runs never trigger it.
- If an error passes through a span, OpenTelemetry marks that span as **ERROR** and marks each parent span as well.

## Section 8 – Execute the RAG Application

**What this section does:** asks three sample IT questions and prints the results.

**What to look for:** each question gets its **own Trace ID**, so each run is stored as a separate trace in Jaeger. The function sends each trace to Jaeger right away, so you do not have to wait for the normal short batch delay (about 5 seconds).

In [ ]:
sample_questions = [
    "How do I reset my password if I am locked out?",
    "My VPN keeps disconnecting. What should I check?",
    "My laptop is running very slowly. What can I do?",
]

for question in sample_questions:
    result = answer_question(question)   # creates one trace for this question
    print("-" * 70)
    print("Question:           ", question)
    print("Documents retrieved:", result["documents_found"])
    print(f"Execution time:      {result['seconds']:.2f} seconds")
    print("Response:           ", result["answer"])
    print()

### What you should see
- Three questions, each with its own **Trace ID** printed above the result. Copy one of these IDs for Section 9.
- **Documents retrieved** is the number of knowledge-base documents sent to the model (at most 3).
- **Execution time** is the total time for that question. The LLM call usually takes most of it.
- Wording may differ from run to run, but each answer should come from the matching documents.

## Section 9 – Explore Traces in Jaeger UI

**What this section does:** guides you through the Jaeger browser page so you can read the traces you just created.

**Step-by-step navigation** (written instructions only; there are no screenshots):
1. Keep this notebook open. In your browser, go to **http://localhost:16686**.
2. On the **Search** page, open the **Service** dropdown and select **module10-rag-app**.
3. Open the **Operation** dropdown and select **RAG Application**. This shows only the top-level run of each question.
4. Click **Find Traces**. Each row is one trace. To find a specific question, paste its **Trace ID** from Section 8 into the **Lookup by Trace ID** box at the top of the page.
5. Click a row to open the trace. The spans appear as a timeline, **indented under their parents**: RAG Application, then User Query, then the four steps.
6. Read the **bars**. The longer the bar, the longer that step took. **LLM Invocation** is usually the longest bar.
7. Click a span to see its **Tags** (its attributes). Look for `retrieval.documents_found`, `gen_ai.request.model` and `gen_ai.usage.input_tokens`.
8. Open a second trace and compare the bars. Which step changed the most?

> Button and menu names can differ slightly between Jaeger versions. If a name is different, choose the closest match.

**Questions to discuss:**
- Which span takes the most time? What would you change first to make the application faster?
- Why is it useful that **User Query** groups the four steps under one parent?

## Section 10 – Error Handling and Debugging Demonstration

**What this section does:** creates one controlled failure. The retrieval step raises an error on purpose. No API call is made, no credentials are used, and no data is changed.

**Why it is required:** in real projects, errors are hard to locate. Tracing shows the exact step that failed, how long the run lasted before it failed, and the error message.

In [ ]:
try:
    answer_question("How do I reset my password?", simulate_failure=True)
except RuntimeError as error:   # the notebook catches the error so the rest of the lab keeps running
    print("The application stopped with this error:", error)
finally:
    tracer_provider.force_flush(timeout_millis=5000)   # make sure the failed trace reaches Jaeger

### How to find the failed trace in Jaeger
1. The notebook printed the **Trace ID** for this run. Paste it into **Lookup by Trace ID** in Jaeger. You can also search with **Tags** set to `error=true`.
2. Open the trace. Three spans are marked as errors: **Document Retrieval**, **User Query** and **RAG Application**.
3. Click **Document Retrieval**. Its details include the message *"Simulated failure: the knowledge base index is unavailable"* and the exception information.
4. Notice that **LLM Invocation** is missing. The failure happened first, so no model call was made and no tokens were used.

**Debugging lesson:** an error travels up through its parent spans. Start with the **deepest span marked as an error**. That is where the fault is.

## Section 11 – Key Learning Outcomes

By completing this lab you can now:
- **Explain OpenTelemetry:** it creates spans in your code and sends them to a tracing backend.
- **Read Jaeger:** it shows each trace as a timeline of parent and child spans.
- **Describe a span:** one step, with a name, timing, attributes and a status.
- **Measure latency:** span durations show which step is slow.
- **Troubleshoot a RAG application:** check retrieval, prompt, model call and response status in order. Start from the deepest error span.
- **Handle trace data securely:** record IDs, counts and timings. Never record secrets, personal data, or full prompts and answers by default.

**Why trace data must be handled securely:** traces are stored and viewed by people and tools. API keys, personal data, and full prompts or responses can leak through them, so production teams keep them out.

## Section 12 – Knowledge Check

Choose the best answer for each question. The answer key is in the last cell of this notebook.

**1. What does observability mean?**
- A. Making an application faster by adding servers
- B. Understanding what an application is doing from the data it produces
- C. Encrypting all user data
- D. Training a new AI model

**2. What is the role of OpenTelemetry?**
- A. Storing documents for a search system
- B. Serving the chat model to users
- C. Starting Docker containers
- D. Creating spans and sending them to a tracing backend

**3. What is Jaeger used for?**
- A. Storing and visualizing traces so you can inspect them
- B. Generating answers from an LLM
- C. Replacing the OpenAI API key
- D. Splitting documents into chunks

**4. In a trace, what does a child span tell you?**
- A. It is always the first span in the trace
- B. It is a separate trace with its own Trace ID
- C. It is a step that ran inside its parent span and shares the parent's Trace ID
- D. It is a log message without timing

**5. A trace shows that Document Retrieval takes 0.01 seconds, but LLM Invocation takes 5.8 seconds. Where should you look first to reduce latency?**
- A. Document Retrieval
- B. LLM Invocation
- C. Prompt Construction
- D. Docker Desktop

## Answer Key

1. **B.** Observability is about understanding what an application does from the data it produces (spans, logs and metrics). It does not by itself make an app faster or train a model.
2. **D.** OpenTelemetry is the open-standard toolkit that creates telemetry such as spans and exports it to a backend like Jaeger.
3. **A.** Jaeger stores traces and shows them in a web interface with timelines, parent-child structure and durations.
4. **C.** A child span is a step inside a parent span. It shares the parent's Trace ID and records the parent's Span ID, which is how Jaeger draws the indentation.
5. **B.** The LLM Invocation bar is by far the longest, so the model call is the main source of latency. Start optimising there (for example, shorter prompts or a faster model), then confirm the change by comparing new traces.